# CKD Counterfactual Demo (Clinically Constrained)

This notebook demonstrates *inference-only* counterfactual generation using the repo artifacts:

- A trained model (default: `models/rf_augmented_42_v1.joblib`)
- The fitted canonical preprocessor (`models/preprocessor.pkl`)

It uses the clinically constrained, gradient-free search implemented in `src/counterfactual.py`.

Key safeguards:

- **Auto-target**: if `target_class=None`, the notebook flips the model's current predicted label (avoids label-convention mismatch).
- **Standardized proximity** (`proximity_mode='zscore'`): distance is normalized by train-set feature std dev (reduces range bias).
- **Pareto selection** (`selection='pareto'`): returns non-dominated CFs over proximity/sparsity/p_target/robustness.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Ensure imports + relative paths work regardless of notebook working directory
PROJECT_ROOT = Path.cwd().resolve()
for _ in range(6):
    if (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'models').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError('Could not locate project root (expected to find ./src and ./models).')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reload to pick up any recent edits to src/counterfactual.py
import importlib
import src.counterfactual as counterfactual  # noqa: E402
importlib.reload(counterfactual)
generate_counterfactual = counterfactual.generate_counterfactual

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

## 1) Load a sample patient

We take one row from the saved test split (`data/processed/splits/X_test_raw.csv`) and pass only the canonical features:

`['hemo', 'sc', 'al', 'htn', 'age', 'dm']`

In [2]:
X_test_raw_path = PROJECT_ROOT / 'data/processed/splits/X_test_raw.csv'
if not X_test_raw_path.exists():
    raise FileNotFoundError('Missing test split. Run: python src/cleaning.py')

X_raw = pd.read_csv(X_test_raw_path)
canonical_cols = ['hemo','sc','al','htn','age','dm']

row_idx = 0  # change this to demo a different patient
patient = X_raw.iloc[row_idx][canonical_cols].to_dict()
patient

{'hemo': 11.4, 'sc': 5.3, 'al': 4.0, 'htn': 1.0, 'age': 71.0, 'dm': 1.0}

## 1b) Optional: auto-pick an “easy” demo patient

Counterfactual search is much faster if the patient is close to the model decision boundary. This cell picks a test patient that is predicted as CKD but already has relatively higher non-CKD probability (typically easiest to flip to non-CKD by improving actionable features).

In [ ]:
# Set this to False if you want to stick to row_idx=0 manually
AUTO_PICK_PATIENT = True

if AUTO_PICK_PATIENT:
    model_path = PROJECT_ROOT / 'models/rf_augmented_42_v1.joblib'
    preprocessor_path = PROJECT_ROOT / 'models/preprocessor.pkl'
    model = counterfactual.load_model(model_path)
    preproc = counterfactual.load_preprocessor(preprocessor_path)

    X_candidates = X_raw[canonical_cols].head(80).copy()
    X_canon = preproc.transform(X_candidates)
    proba = model.predict_proba(X_canon)
    pred = model.predict(X_canon)
    classes = getattr(model, 'classes_', None)
    if classes is None or len(classes) != proba.shape[1]:
        raise RuntimeError('Model classes_ missing or inconsistent with predict_proba output.')

    # Prefer picking a CKD-predicted patient and target non-CKD (label 0) if present
    classes_list = [int(x) for x in classes]
    if 0 in classes_list and 1 in classes_list:
        idx_nonckd = classes_list.index(0)
        idx_ckd = classes_list.index(1)
        mask_ckd = (pred.astype(int) == 1)
        if mask_ckd.any():
            # choose CKD cases that already have higher non-CKD probability (closest to boundary)
            cand_idx = (proba[mask_ckd, idx_nonckd]).argmax()
            row_idx = int(X_candidates.index[mask_ckd][cand_idx])
        else:
            # fallback: just choose the most uncertain sample overall
            margin = (proba[:, idx_ckd] - proba[:, idx_nonckd]).astype(float)
            row_idx = int(X_candidates.index[abs(margin).argmin()])
    else:
        # Generic fallback: pick the most uncertain sample (smallest top-2 margin)
        sorted_p = np.sort(proba, axis=1)[:, ::-1]  # descending sort per row
        margin = (sorted_p[:, 0] - sorted_p[:, 1]).astype(float)
        row_idx = int(X_candidates.index[int(np.argmin(margin))])

    patient = X_raw.loc[row_idx, canonical_cols].to_dict()
    print('AUTO_PICK row_idx =', row_idx)
    print('patient =', patient)

AUTO_PICK row_idx = 68
patient = {'hemo': 14.3, 'sc': 0.8, 'al': 0.0, 'htn': 0.0, 'age': 14.0, 'dm': 1.0}


d:\Anaconda\envs\ckd-shap50\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Anaconda\envs\ckd-shap50\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 2) Generate counterfactuals

Recommended live-demo settings (to flip **CKD → NotCKD**):

- `target_class=0` (explicitly target NotCKD; in this repo `ckd=1`, `notckd=0`)
- `selection='pareto'` (returns Pareto-optimal set)
- `proximity_mode='zscore'` (standardized proximity)
- `max_iter=80`, `target_prob=0.80` (usually fast enough for a demo; adjust if needed)

Note: you *can* set `target_class=None` to auto-flip the model’s current predicted label, but if you accidentally pick a patient that is already NotCKD, auto-flip will target CKD (the opposite of what we want here).

In [ ]:
# Paths (explicit, so it works regardless of current working directory)
model_path = PROJECT_ROOT / 'models/rf_augmented_42_v1.joblib'
preprocessor_path = PROJECT_ROOT / 'models/preprocessor.pkl'

if not model_path.exists():
    raise FileNotFoundError(f'Model not found: {model_path}')
if not preprocessor_path.exists():
    raise FileNotFoundError(f'Preprocessor not found: {preprocessor_path}. Run: python src/cleaning.py')

# Sanity-check the original prediction (in this repo: 1=CKD, 0=NotCKD)
_model = counterfactual.load_model(model_path)
_preproc = counterfactual.load_preprocessor(preprocessor_path)
_x0 = counterfactual.prepare_patient_canonical(patient, preprocessor=_preproc, preprocess=True)
print('model classes_:', getattr(_model, 'classes_', None))
print('original predicted label:', int(_model.predict(_x0)[0]))
print('original proba:', counterfactual.predict_proba_cf(_model, _x0)[0])

# Demo settings tuned for speed + high chance of success
weights = counterfactual.LossWeights(lambda_rob=0.0)

cfs_df, metrics_df, explanation, comparison = generate_counterfactual(
    patient,
    model_path=model_path,
    preprocessor_path=preprocessor_path,
    target_class=0,  # NotCKD
    k=2,
    max_iter=60,
    n_neighbors=12,
    pool_size=4,
    max_attempts=25,
    target_prob=0.60,
    proximity_mode='zscore',
    selection='pareto',
    compute_explanation=False,
    explanation_stability_runs=0,
    seed=42,
    weights=weights,
 )

print('auto_target:', explanation.get('auto_target'))
print('resolved target label:', explanation.get('target_class'))
print('resolved target index:', explanation.get('target_index'))

metrics_df

d:\Anaconda\envs\ckd-shap50\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Anaconda\envs\ckd-shap50\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


auto_target: True
resolved target label: 0
resolved target index: 0


,validity,proximity,sparsity,p_target,robustness,pareto_optimal
0,True,2.353436,4,0.962494,1.0,True
1,True,2.198328,2,0.960881,1.0,True


## 2b) Optional: compute SHAP explanation (run only if needed)

The counterfactual search above can be slow if it also computes SHAP. This optional step computes SHAP **after** the counterfactual is selected.

In [16]:
import pandas as pd

if metrics_df.empty:
    print('No valid CF found; skipping SHAP explanation.')
else:
    # Pick the best CF by highest p_target then lowest proximity
    best_row = metrics_df.sort_values(by=['p_target', 'proximity'], ascending=[False, True]).index[0]
    best_cf = cfs_df.loc[[best_row]][['hemo','sc','al','htn','age','dm']].copy()
    x_original = comparison[comparison['row'] == 'original'][['hemo','sc','al','htn','age','dm']].copy()
    # Replace counterfactual row with our selected best_cf (same schema)
    x_cf_best = best_cf.copy()

    model = counterfactual.load_model(model_path)
    target_label = int(explanation.get('target_class'))
    target_index = int(explanation.get('target_index'))

    explanation = counterfactual.explain_original_vs_counterfactual(
        model=model,
        x_original=x_original,
        x_cf_best=x_cf_best,
        target_label=target_label,
        target_index=target_index,
        stability_runs=0,
        rng=None,
    )
    explanation['auto_target'] = True
    print('SHAP available:', explanation.get('shap_available'))

d:\Anaconda\envs\ckd-shap50\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Anaconda\envs\ckd-shap50\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Anaconda\envs\ckd-shap50\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/use

SHAP available: True


## 3) View the counterfactual(s) and comparison table

In [17]:
display(cfs_df)

comparison

,hemo,sc,al,htn,age,dm
0,14.497566,0.4,0.044841,0.0,14.0,0.0
1,14.492545,0.8,0.000000,0.0,14.0,0.0


,row,hemo,sc,al,htn,age,dm
0,original,14.300000,0.8,0.000000,0.0,14.0,1.0
1,best_counterfactual,14.497566,0.4,0.044841,0.0,14.0,0.0


## 4) Explanation summary

- `clinical_text` is a human-readable change summary.
- If SHAP is installed and compatible with the model, you will also see SHAP deltas and a top-k overlap metric.

In [18]:
print(explanation.get('clinical_text', explanation.get('note', 'No clinical text available.')))

# Show key explanation fields (if SHAP ran)
keys_to_show = [
    'original_proba', 'counterfactual_proba',
    'top_features_changed',
    'top_shap_delta_features',
    'shap_topk_overlap',
    'shap_input_stability',
]
{k: explanation.get(k) for k in keys_to_show if k in explanation}

No clinical text available.


{'original_proba': [0.22519241486831748, 0.7748075851316826],
 'counterfactual_proba': [0.9624943214937978, 0.037505678506202525],
 'top_shap_delta_features': ['hemo', 'sc'],
 'shap_topk_overlap': {'k': 5,
  'jaccard': 1.0,
  'features_original': ['hemo', 'sc'],
  'features_counterfactual': ['hemo', 'sc']}}

## Notes / troubleshooting

1. If you get `No valid counterfactual found...`, try: 
   - lowering `target_prob` (e.g., 0.75), or
   - increasing `max_iter` (e.g., 150–300), or
   - increasing `k` and letting Pareto selection pick good candidates.

2. If SHAP is missing/unsupported, the pipeline still works; the explanation dict will include a note.

3. The search is heuristic and may converge to local optima; the implementation uses restarts to reduce that risk.